# Track A — Bước A5: `DINOv1` độ phân giải cao `448×448` (`interpolate_pos_encoding=True`) trên `GC10-DET`

- Nhánh: `duytung/09-gc10-dinov1-highres`
- Mục tiêu: Giữ nguyên `DINOv1` (`facebook/dino-vitb16`), tăng độ phân giải đầu vào từ `224×224` lên **`448×448`** (`28×28 = 784` patch, gấp 4 lần số patch) để bảo toàn chi tiết các vết lỗi nhỏ trên ảnh thép `2048×1000` (không cần nhãn XML, không cần `HF_TOKEN`).
- Dùng `FT_BATCH_SIZE=8, FT_ACCUM=16` (effective batch = `128` giống A0) để an toàn VRAM trên cả T4 và L4.
- Tự động đo Peak CPU/RAM/GPU/VRAM và xuất ra `gc10_a5_highres_summary.txt` / `.json` trên Drive.

In [ ]:
# 1. Kết nối Google Drive (bỏ qua nếu runtime hiện tại đã mount sẵn)
import os
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Google Drive đã được mount sẵn!')

In [ ]:
# 2. Cập nhật nhánh duytung/09-gc10-dinov1-highres
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'
BRANCH = 'duytung/09-gc10-dinov1-highres'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout -B {BRANCH} origin/{BRANCH}
!git status -sb

In [ ]:
# 3. Cài gói thiếu & kiểm tra GPU (bỏ qua nếu runtime hiện tại đã cài)
!pip uninstall -y -q torchao
!pip install -q numpyro continuum peft datasets transformers scipy scikit-learn psutil
!nvidia-smi

In [ ]:
# 4. Chạy nền Bước A5 (DINOv1 448x448, FT_EPOCHS=20, r0, s0..s4)
import os, subprocess, time
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a5_highres.log'
PID_FILE = f'{WORK_DIR}/gc10_a5_highres.pid'

with open(LOG_FILE, 'w') as f:
    p = subprocess.Popen(['bash', 'colab/run_gc10_a5_highres_bg.sh', 'auto'], stdout=f, stderr=subprocess.STDOUT)
with open(PID_FILE, 'w') as f:
    f.write(str(p.pid))
print(f'Đã khởi chạy nền Bước A5 (DINOv1 448x448) với PID={p.pid}')
print(f'Log: {LOG_FILE}')
time.sleep(3)
!tail -n 20 {LOG_FILE}

In [ ]:
# 5. Kiểm tra tiến độ & xem kết quả + thông số phần cứng peak
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
LOG_FILE = f'{WORK_DIR}/gc10_a5_highres.log'
PID_FILE = f'{WORK_DIR}/gc10_a5_highres.pid'
SUMMARY_OUT = f'{WORK_DIR}/gc10_a5_highres_summary.txt'

pid = open(PID_FILE).read().strip() if os.path.exists(PID_FILE) else ''
alive = False
if pid and os.path.exists(f'/proc/{pid}/status'):
    st = open(f'/proc/{pid}/status').read()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}: {"ĐANG CHẠY" if alive else "ĐÃ DỪNG / XONG"}')
print('\n--- 40 dòng log gần nhất ---')
if os.path.exists(LOG_FILE):
    lines = open(LOG_FILE).read().splitlines()
    print('\n'.join(lines[-40:]))
if os.path.exists(SUMMARY_OUT):
    print('\n--- BẢNG KẾT QUẢ & THÔNG SỐ PHẦN CỨNG PEAK BƯỚC A5 (GC10 DINOv1 448x448) ---')
    print(open(SUMMARY_OUT).read())